In [1]:
import os, time, random, math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")




In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)



Device: cuda


In [3]:
DATA_PATH = "./cache_qm9/qm9_splits_vocab.pt"  # adjust if needed
obj = torch.load(DATA_PATH)
train_sm, val_sm, test_sm = obj["splits"]
stoi, itos = obj["stoi"], obj["itos"]

PAD, BOS, EOS = "<PAD>", "<BOS>", "<EOS>"
pad_id = stoi[PAD]
bos_id = stoi[BOS]
eos_id = stoi[EOS]
vocab_size = len(itos)

print("Loaded:", len(train_sm), len(val_sm), len(test_sm), "Vocab:", vocab_size)

Loaded: 120496 6694 6695 Vocab: 24


In [4]:
MAX_LEN = 80  

def encode(sm: str, stoi: Dict[str,int], max_len: int) -> List[int]:
    ids = [stoi[BOS]] + [stoi[c] for c in sm] + [stoi[EOS]]
    if len(ids) < max_len:
        ids = ids + [stoi[PAD]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
        ids[-1] = stoi[EOS]
    return ids

def decode(ids: List[int], itos: List[str]) -> str:
    stoi_local = {c:i for i,c in enumerate(itos)}
    out = []
    for i in ids:
        if i == stoi_local[EOS]:
            break
        if i in (stoi_local[PAD], stoi_local[BOS]):
            continue
        out.append(itos[i])
    return "".join(out)

class SmilesVAEDataset(Dataset):
    def __init__(self, smiles_list: List[str], stoi: Dict[str,int], max_len: int):
        self.smiles = smiles_list
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode(self.smiles[idx], self.stoi, self.max_len)  # length MAX_LEN
        x = torch.tensor(ids, dtype=torch.long)                  # (T)
        return x

train_ds = SmilesVAEDataset(train_sm, stoi, MAX_LEN)
val_ds   = SmilesVAEDataset(val_sm, stoi, MAX_LEN)

BATCH_SIZE = 256 if DEVICE=="cuda" else 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(DEVICE=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEVICE=="cuda"))

print("Batches:", len(train_loader), len(val_loader))


Batches: 471 27


In [5]:
@dataclass
class VAEConfig:
    vocab_size: int
    emb_dim: int = 256
    enc_hidden: int = 512
    dec_hidden: int = 512
    num_layers: int = 1
    z_dim: int = 64
    dropout: float = 0.0

class SmilesVAE(nn.Module):
    def __init__(self, cfg: VAEConfig, pad_id: int, bos_id: int, eos_id: int):
        super().__init__()
        self.cfg = cfg
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id

        self.embed = nn.Embedding(cfg.vocab_size, cfg.emb_dim, padding_idx=pad_id)

        # Encoder: GRU over full sequence
        self.enc_gru = nn.GRU(
            input_size=cfg.emb_dim,
            hidden_size=cfg.enc_hidden,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=cfg.dropout if cfg.num_layers > 1 else 0.0
        )

        # Latent parameters
        self.to_mu = nn.Linear(cfg.enc_hidden, cfg.z_dim)
        self.to_logvar = nn.Linear(cfg.enc_hidden, cfg.z_dim)

        # Decoder initial hidden from z
        self.z_to_h0 = nn.Linear(cfg.z_dim, cfg.dec_hidden * cfg.num_layers)

        # Decoder GRU
        self.dec_gru = nn.GRU(
            input_size=cfg.emb_dim,
            hidden_size=cfg.dec_hidden,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=cfg.dropout if cfg.num_layers > 1 else 0.0
        )

        self.fc_out = nn.Linear(cfg.dec_hidden, cfg.vocab_size)

    def encode(self, x):
        # x: (B, T)
        emb = self.embed(x)                # (B, T, E)
        _, h = self.enc_gru(emb)           # h: (L, B, H)
        h_last = h[-1]                     # (B, H)
        mu = self.to_mu(h_last)            # (B, z)
        logvar = self.to_logvar(h_last)    # (B, z)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        # z = mu + sigma * eps
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z, x_in):
        """
        z: (B, z_dim)
        x_in: decoder input tokens (B, T-1) (teacher forcing)
        """
        B = z.size(0)
        L = self.cfg.num_layers

        h0 = self.z_to_h0(z).view(L, B, self.cfg.dec_hidden).contiguous()  # (L,B,H)
        emb = self.embed(x_in)                                             # (B,T-1,E)
        out, _ = self.dec_gru(emb, h0)                                     # (B,T-1,H)
        logits = self.fc_out(out)                                          # (B,T-1,V)
        return logits

    def forward(self, x):
        """
        x: (B, T) full tokens including BOS/EOS/PAD
        We train decoder to predict next token: x[:, 1:] from inputs x[:, :-1]
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)

        x_in = x[:, :-1]       # (B, T-1)
        x_tgt = x[:, 1:]       # (B, T-1)

        logits = self.decode(z, x_in)  # (B, T-1, V)
        return logits, x_tgt, mu, logvar


In [6]:
def kl_divergence(mu, logvar):
    # KL(q(z|x) || N(0,I)) for diagonal Gaussian
    return 0.5 * torch.sum(torch.exp(logvar) + mu**2 - 1.0 - logvar, dim=1)  # (B,)

def recon_loss(logits, targets, pad_id: int):
    # logits: (B, T-1, V), targets: (B, T-1)
    B, T, V = logits.shape
    return F.cross_entropy(logits.reshape(B*T, V), targets.reshape(B*T), ignore_index=pad_id)

vae_cfg = VAEConfig(vocab_size=vocab_size, z_dim=64)
vae = SmilesVAE(vae_cfg, pad_id, bos_id, eos_id).to(DEVICE)

optimizer = torch.optim.AdamW(vae.parameters(), lr=2e-3, weight_decay=1e-2)

def run_epoch(loader, train: bool, kl_weight: float):
    vae.train(train)
    total = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    n = 0
    for x in loader:
        x = x.to(DEVICE)
        logits, tgt, mu, logvar = vae(x)
        r = recon_loss(logits, tgt, pad_id)
        kl = kl_divergence(mu, logvar).mean()
        loss = r + kl_weight * kl

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            optimizer.step()

        total["loss"] += loss.item()
        total["recon"] += r.item()
        total["kl"] += kl.item()
        n += 1

    for k in total:
        total[k] /= max(1, n)
    return total


In [7]:
EPOCHS = 5
for epoch in range(1, EPOCHS+1):
    kl_w = min(1.0, epoch / max(1, EPOCHS//2))
    t0 = time.time()
    tr = run_epoch(train_loader, train=True, kl_weight=kl_w)
    va = run_epoch(val_loader, train=False, kl_weight=kl_w)
    t1 = time.time()
    print(f"Epoch {epoch:02d} | kl_w={kl_w:.2f} | "
          f"train L={tr['loss']:.3f} (R={tr['recon']:.3f}, KL={tr['kl']:.3f}) | "
          f"val L={va['loss']:.3f} (R={va['recon']:.3f}, KL={va['kl']:.3f}) | {t1-t0:.1f}s")


Epoch 01 | kl_w=0.50 | train L=0.952 (R=0.952, KL=0.000) | val L=0.843 (R=0.843, KL=0.000) | 14.1s
Epoch 02 | kl_w=1.00 | train L=0.832 (R=0.832, KL=0.000) | val L=0.823 (R=0.823, KL=0.000) | 13.3s
Epoch 03 | kl_w=1.00 | train L=0.817 (R=0.817, KL=0.000) | val L=0.814 (R=0.814, KL=0.000) | 13.3s
Epoch 04 | kl_w=1.00 | train L=0.808 (R=0.808, KL=0.000) | val L=0.812 (R=0.812, KL=0.000) | 13.3s
Epoch 05 | kl_w=1.00 | train L=0.803 (R=0.803, KL=0.000) | val L=0.806 (R=0.806, KL=0.000) | 13.3s


In [8]:
@torch.no_grad()
def sample_vae(vae: SmilesVAE, n: int, max_len: int, temperature: float = 1.0) -> List[str]:
    vae.eval()
    samples = []

    z = torch.randn(n, vae.cfg.z_dim, device=DEVICE)

    L = vae.cfg.num_layers
    h = vae.z_to_h0(z).view(L, n, vae.cfg.dec_hidden).contiguous()

    cur = torch.full((n, 1), bos_id, dtype=torch.long, device=DEVICE)
    out_ids = [[] for _ in range(n)]

    for t in range(max_len - 1):
        emb = vae.embed(cur)             # (n,1,E)
        out, h = vae.dec_gru(emb, h)     # out: (n,1,H)
        logits = vae.fc_out(out[:, -1, :]) / max(1e-8, temperature)  # (n,V)
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1)  # (n,1)

        nxt_ids = nxt.squeeze(1).tolist()
        for i, nid in enumerate(nxt_ids):
            if nid == eos_id:
                continue
            if nid not in (pad_id, bos_id):
                out_ids[i].append(nid)

        cur = nxt

    for i in range(n):
        sm = "".join(itos[j] for j in out_ids[i])
        samples.append(sm)
    return samples

# smoke test
smoke = sample_vae(vae, n=10, max_len=MAX_LEN, temperature=0.9)
smoke[:5]


['C#CC1(N)CN1C(N)=O',
 'N#CC1NC2(CC2)C1',
 'CN=C1C(O)C2NC12',
 'O=C1CC2NC1C2CO',
 'CC(O)C1C(C)C1(C)C2']

In [9]:
def to_mol(smiles: str):
    return Chem.MolFromSmiles(smiles)

def canonical(smiles: str) -> str | None:
    m = to_mol(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m)

train_canon: Set[str] = set()
for s in train_sm:
    cs = canonical(s)
    if cs is not None:
        train_canon.add(cs)

def validity(smiles_list: List[str]) -> Tuple[float, List[str]]:
    valid_canon = []
    for s in smiles_list:
        cs = canonical(s)
        if cs is not None:
            valid_canon.append(cs)
    return len(valid_canon) / max(1, len(smiles_list)), valid_canon

def uniqueness(valid_canon: List[str]) -> float:
    return 0.0 if len(valid_canon)==0 else len(set(valid_canon))/len(valid_canon)

def novelty(valid_canon: List[str], train_set: Set[str]) -> float:
    return 0.0 if len(valid_canon)==0 else sum(1 for s in valid_canon if s not in train_set)/len(valid_canon)

def compute_properties(valid_canon: List[str]) -> Dict[str,float]:
    if len(valid_canon)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    mws, logps, qeds = [], [], []
    for s in valid_canon:
        m = to_mol(s)
        if m is None:
            continue
        mws.append(Descriptors.MolWt(m))
        logps.append(Crippen.MolLogP(m))
        qeds.append(QED.qed(m))
    if len(mws)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    return {"mw_mean": sum(mws)/len(mws), "logp_mean": sum(logps)/len(logps), "qed_mean": sum(qeds)/len(qeds)}

def evaluate_smiles(samples: List[str], train_set: Set[str]) -> Dict[str,float]:
    v, valid_canon = validity(samples)
    out = {
        "n_samples": len(samples),
        "validity": v,
        "n_valid": len(valid_canon),
        "uniqueness": uniqueness(valid_canon),
        "novelty": novelty(valid_canon, train_set),
    }
    out.update(compute_properties(valid_canon))
    return out


In [10]:
N_SAMPLES = 5000

t0 = time.time()
gen = sample_vae(vae, n=N_SAMPLES, max_len=MAX_LEN, temperature=0.9)
t1 = time.time()

metrics_vae = evaluate_smiles(gen, train_canon)
gen_seconds = t1 - t0

metrics_vae["gen_seconds"] = gen_seconds
metrics_vae["samples_per_sec"] = N_SAMPLES / gen_seconds
metrics_vae["valid_per_sec"] = metrics_vae["n_valid"] / gen_seconds

metrics_vae


{'n_samples': 5000,
 'validity': 0.918,
 'n_valid': 4590,
 'uniqueness': 0.9671023965141612,
 'novelty': 0.23311546840958605,
 'mw_mean': 123.4424934640518,
 'logp_mean': 0.4992854880174304,
 'qed_mean': 0.47733036436912024,
 'gen_seconds': 0.1456608772277832,
 'samples_per_sec': 34326.30707132872,
 'valid_per_sec': 31511.54989147977}

In [11]:
os.makedirs("./checkpoints", exist_ok=True)
ckpt_path = "./checkpoints/gru_vae_qm9_z64.pt"

torch.save({
    "model_state": vae.state_dict(),
    "vae_cfg": vae_cfg.__dict__,
    "stoi": stoi,
    "itos": itos,
    "max_len": MAX_LEN,
}, ckpt_path)

print("Saved:", ckpt_path)

with open("./checkpoints/gru_vae_samples.txt", "w") as f:
    for s in gen[:200]:
        f.write(s + "\n")
print("Saved sample text file.")


Saved: ./checkpoints/gru_vae_qm9_z64.pt
Saved sample text file.
